# EJERCICIO 
# Aprendizaje no supervisado: Clustering.

## *Segmentación de clientes*


En el aprendizaje no supervisado, la tarea clásica es el **análisis de clusters** (grupos) en el que se encuentran patrones o grupos ocultos en los datos. La mayoría de las veces las tareas de aprendizaje no supervisado tienen una *solución abierta*, por lo que hay que interpretar los resultados y comprobar si tienen sentido.

**Objetivo:** En este ejemplo, se realizará un clustering de datos sobre clientes de una empresa de comestibles. La segmentación de clientes es la práctica de separar a los clientes en grupos que reflejen similitudes. Se dividirá a los clientes en segmentos para optimizar la importancia de cada uno de ellos. Así, la empresa podrá modificar los productos en función de las distintas necesidades y comportamientos de los clientes. También ayuda a la empresa a atender las preocupaciones de distintos tipos de clientes.

<img src="Data/market_segmentation.png" alt="market_segmentation" class="bg-primary" width="800px">

### Antes de empezar:

* En el archivo **marketing_campaign.csv** se encuentra el conjunto de datos de entrada de este ejemplo (atributos). 
* **NO** existen las etiquetas en el Aprendizaje **NO Supervisado**. 

## **1. Importar librerías y datos**

In [ ]:
# Importar librerías
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## **2. Comprender los datos**

Es necesario visualizar y comprender los datos con los que vamos a trabajar, así como conocer sus características. 

1. ¿Cuántos datos hay? ¿Cuántos atributos hay en los datos?  
2. ¿Qué significan?
3. ¿Falta algún dato?
4. Resumen estadístico del conjunto de datos de entrada.

In [ ]:
# Primera visualización del dataset
df_clientes = pd.read_csv('Data/marketing_campaign.csv', sep="\t")
df_clientes.head()

**1. ¿Cuántos datos hay? ¿Cuántos atributos hay en los datos?**

In [ ]:
# Dimensión de los datos de entrada (filas x columnas)
################ CÓDIGO AQUÍ ########################

In [ ]:
# Ya hemos visto el "head", vamos a ver el "tail"
df_clientes.tail()

**2. ¿Qué significan?**

In [ ]:
# Información sobre el dataset: https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis
# También podemos obtener el nombre de las columnas
df_clientes.columns

In [ ]:
# Se comprueba el tipo de variable de cada columna
df_clientes.dtypes

**3. Resumen estadístico del conjunto de datos de entrada:** La estadística descriptiva recolecta y analiza el conjunto de datos de entrada con el objetivo de describir las características y comportamientos de este conjunto mediante las siguientes medidas resumen: número total de observaciones (count), media (mean), desviación estándar (std), valor mínimo (min), valor máximo (max) y los valores de los diferentes cuartiles (25%, 50%, 75%).

In [ ]:
# Evaluamos la naturaleza de los datos con datos estadísticos descriptivos
################ CÓDIGO AQUÍ ########################

**4. ¿Falta algún dato?** Se comprueba si falta algún dato, y de ser así, se realiza el recuento de celdas vacías en cada atributo.

In [ ]:
################ CÓDIGO AQUÍ ########################

## **3. Limpieza de datos**

Antes de empezar a trabajar con los datos, estos se tienen que limpiar (anteriormente ya se han eliminado los NaNs) y transformar en otros features en caso de ser necesario 

In [ ]:
df_clientes.info()

De la función `df_clientes.info()` vemos que no tenemos ningun valor nulo, aunque en la columna 'Income' tenemos 2216 muestras en vez de 2240. También vemos que la columna 'Dt_Customer' está en formato `object` en vez de `DateTime` y que hay 2 columnas más con formato `object`, en las cuales tendremos que codificar para que el código pueda entenderlas.

**¿Qué hacemos con los NaN?**

In [ ]:
# Una solución fácil para este dataset (>2000 filas) seria eliminar las filas con valores NaN
# Creamos una copia de nuestro df para no perder el dataset inicial
df_clean = df_clientes.copy()
################ CÓDIGO AQUÍ ########################
# Otras opciones?

In [ ]:
# Comprobamos que ya no faltan más valores
df_clean.isna().sum()

**Queremos que *Dt_Customer* esté en formato datetime.** Este formato nos permitirá aplicar funciones para identificar el dia/mes/año y otros métodos específicos para este tipo de variables.

In [ ]:
df_clean["Dt_Customer"]
# Vemos que los datos están en formato DD-MM-YYYY. Tenemos que especificarlo para que la función entienda como formatear los datos.

In [ ]:
# Cambiamos el formato de la columna 'Dt_Customer' a DateTime
df_clean["Dt_Customer"] = pd.to_datetime(df_clean["Dt_Customer"], format="%d-%m-%Y")

# Para que tenga más sentido, cambiamos la fecha en la que una persona se hizo cliente por una variable que cuente
# los días que esa persona lleva siendo cliente. e.g.: ["Customer_For"]
fechas = ################ CÓDIGO AQUÍ ########################
ultima_fecha = ################ CÓDIGO AQUÍ ########################
df_clean["Customer_For_days"] = (ultima_fecha - df_clean["Dt_Customer"]).dt.days

df_clean.head()

**Las columnas con formato *object* son datos categóricos y tienen que tranformarse a valores numéricos**. ¿Cómo lo hacemos?

In [ ]:
print("Total categories in the feature Marital_Status:\n", df_clean["Marital_Status"].value_counts(), "\n")
print("Total categories in the feature Education:\n", df_clean["Education"].value_counts())

***Marital_Status.*** Opciones:
- Integer encoding.
- One-hot encoding (en 1 o múltiples columnas).

In [ ]:
# Primero reducimos el Martial_Status a 2 variables (pareja/solo)
df_clean["Living_With"]=df_clean["Marital_Status"].replace({"Married":"Partner", "Together":"Partner", "Absurd":"Alone", "Widow":"Alone", "YOLO":"Alone", "Divorced":"Alone", "Single":"Alone",})

# Ahora hacemos one-hot encoding creando la columna 'Partner'
df_clean["Partner"] = df_clean['Living_With'].map({"Alone": 0, "Partner":1})

# Eliminamos las columnas "Marital_Status" y "Partner"
df_clean.drop(columns=['Marital_Status', 'Living_With'], inplace=True)
df_clean

***Education***. Opciones:
- Integer encoding.
- One-hot encoding (en 1 o múltiples columnas).

In [ ]:
# Podemos mapear directamente las variables usando integer encoding
df_clean["Education_int"] = ################ CÓDIGO AQUÍ ########################

# Eliminamos la columna "Education"
df_clean.drop(columns=['Education'], inplace=True)
df_clean

**De las variables numéricas también podemos derivar características informativas para nuestro modelo**

A continuación, realizaremos los siguientes pasos para crear algunos Features nuevos:

* **"Year_Birth"** no es muy informativo: Extraemos la edad **("Age")** del cliente.
* Crear otra función **"Spent"** que indique la cantidad total gastada por el cliente en varias categorías en un periodo de dos años.
* Crear una función **"Children"** para indicar el total de hijos de un hogar, es decir, niños **("Kidhome")** y adolescentes **("Teenhome")**.
* Para obtener una mayor claridad del hogar, crear una característica que indique **"Family_Size"**.
* Crear una característica **"Is_Parent"** para indicar la situación de paternidad
* Tambien eliminaremos algunas de las características redundantes

In [ ]:
# Feature Engineering
# Age of customer today 
ultimo_año = ultima_fecha.year
df_clean["Age"] = ################ CÓDIGO AQUÍ ########################

# Total spendings on various items
df_clean["Spent"] = df_clean["MntWines"]+ df_clean["MntFruits"]+ df_clean["MntMeatProducts"]+ df_clean["MntFishProducts"]+ df_clean["MntSweetProducts"]+ df_clean["MntGoldProds"]

# Feature indicating total children living in the household
df_clean["Children"] = ################ CÓDIGO AQUÍ ########################

# Feature for total members in the household
df_clean["Family_Size"] = ################ CÓDIGO AQUÍ ########################

# Feature pertaining parenthood
df_clean["Is_Parent"] = ################ CÓDIGO AQUÍ ########################

# Cambiamos algunos nombres de columnas
df_cluster = df_clean.rename(columns={"MntWines": "SpentWines","MntFruits":"SpentFruits","MntMeatProducts":"SpentMeat",
                                      "MntFishProducts":"SpentFish","MntSweetProducts":"SpentSweets","MntGoldProds":"SpentGold"})

# Dropping some of the redundant features
to_drop = ["Dt_Customer", "Z_CostContact", "Z_Revenue", "Year_Birth", "ID"]
df_cluster = df_cluster.drop(to_drop, axis=1)
df_cluster

In [ ]:
df_cluster.dtypes

Ahora, vamos a visualizar los datos para tener más información de ellos.

**1. Histograma**

Respresentación gráfica de cada uno de los atributos en forma de barras, donde la superficie de la barra es proporcional a la frecuencia de los valores representados.

In [ ]:
histograma = df_cluster.hist(xlabelsize=10, ylabelsize=10, bins=100, figsize=(18, 12))

Algunas categorias (como Income y Age) parecen tener outliers. Vamos a comprobarlo con un boxplot y en caso de que sea necesario, eliminamos los outliers.

In [ ]:
boxplot = df_cluster.boxplot(column=['Income'])

In [ ]:
boxplot = df_cluster.boxplot(column=['Age'])

In [ ]:
# Tiene sentido eliminar los outliers de estas columnas?
# Filtramos los usuarios con más de 100 años, asumiendo que son anomalías
# Podemos hacer lo mismo con 'Income', si lo creemos conveniente
df_cluster = ################ CÓDIGO AQUÍ ########################
df_cluster.reset_index(drop=True, inplace=True)
df_cluster

Ahora, vamos a mirar la correlación entre variables (excluyendo los atributos categoricos).

In [ ]:
#correlation matrix
corrmat= df_cluster.corr()
plt.figure(figsize=(20,20))  
sns.heatmap(corrmat, annot=True, cmap='coolwarm', center=0)

In [ ]:
# Matriz de correlación solo con respecto a 'Spent'
corr_with_spent = df_cluster.corr()['Spent'].sort_values(ascending=False).to_frame()

plt.figure(figsize=(6, 12))
sns.heatmap(corr_with_spent, annot=True, cmap='coolwarm', center=0)
plt.title("Correlación con 'Spent'")
plt.show()

## *4. Procesamiento de los datos*

1. Si no lo hemos hecho antes, encoding para las variables categóricas.
2. Escalar los datos usando el Standard Scaler.

**1. Encoding para las variables categóricas**. 

In [ ]:
# Obtener lista de variables categóricas
cat = (df_cluster.dtypes == 'object')
cols_cat = list(cat[cat].index)
cols_cat

print("Variables categoricas en el dataset:", cols_cat)

**Paso extra (opcional): reducimos el número de columnas para entender mejor el algoritmo**

Nos quedaremos solo con las siguientes columnas: 'Income', 'Spent', 'Age', 'Customer_For_days', 'Kidhome', 'NumCatalogPurchases', 'NumWebVisitsMonth' y 'Family_Size'

In [ ]:
df_cluster_reduced = df_cluster.copy()
# df_cluster_reduced = df_cluster[['Income', 'Spent', 'Age', 'Customer_For_days', 'Kidhome', 'NumCatalogPurchases', 'NumWebVisitsMonth', 'Family_Size']].copy()
df_cluster_reduced.head()

**2. Escalar los datos**. 

Se escalan los datos utilizando el método de *StandardScaler()*

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = df_cluster_reduced.values.copy()
X_scale = pd.DataFrame(scaler.fit_transform(X))
X_scale

## 5. Construcción del modelo de aprendizaje NO supervisado: Clustering de consumos utilizando K-means

Se agrupan los datos utilizando el algoritmo [K-Means](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).

El algoritmo K-means necesita que se le indique el número de clústers en que se quieren agrupar los datos. Se ejecuta el algoritmo para varios clusters y luego se comparan los resultados utilizando el método Elbow, que indicará el número óptimo de clusters.

### ¿Cómo saber el número óptimo de clusters? Con el método de Elbow.
Se utiliza el [Método de Elbow] para ayudarnos a elegir el número óptimo de clusters. 

* Este método utiliza los valores de la inercia obtenidos tras aplicar el K-means a diferente número de Clusters (desde 1 a N Clusters), siendo la inercia la suma de las distancias al cuadrado de cada objeto del Cluster a su centroide.
* Para hacer uso de este método partimos del cálculo de la distorsión promedio de cada clúster, esto es la distancia de cada elemento con su centroide correspondiente.
* Buscamos la parte de la gráfica donde la línea es menos suave o cambia abruptamente lo que forma un “codo”.

[Método de Elbow]: https://jarroba.com/seleccion-del-numero-optimo-clusters/


<img src="Data/elbow_method.png" alt="elbow" class="bg-primary" width="400px">

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

elbow_method = []

# Evalúo el algoritmo K-means para un rango de [1,15] clústers 
n_cluster_list = range(1,15)

In [ ]:
import matplotlib.pyplot as plt

# Iteración para evaluar K-means para diferentes números de clusters (n_clusters)
for n_cluster in n_cluster_list:
    kmeans = KMeans(n_clusters=n_cluster, random_state=0)
    cluster_found = kmeans.fit_predict(X_scale)
    elbow_method.append(kmeans.inertia_) 


# Gráfica del método de Elbow
plt.plot(n_cluster_list, elbow_method, 'b', marker='*')
plt.xlabel('Número de centroides')
plt.ylabel('Within-Cluster Sum of Square')
plt.title('Método Elbow para número óptimo de clusters')
plt.show()

# El número óptimo de clusters es de k=2, aunque podria ser interesante usar k=3 o incluso k=6

In [ ]:
# Entreno el K-means para k=2, visto el resultado del método Elbow
kmeans = KMeans(n_clusters=2)
cluster_found = kmeans.fit_predict(X_scale)

df_cluster_def = df_cluster_reduced.copy()
df_cluster_def['cluster'] = pd.Series(cluster_found)
df_cluster_def

In [ ]:
# Hacemos scatter plot entre 'Income' y 'Spent'
plt.figure(figsize=(7,7))
sns.scatterplot(x=df_cluster_def["Income"], y=df_cluster_def["Spent"], hue=df_cluster_def["cluster"])

# Estética del plot
plt.title('Scatter Plot', fontsize=12)
plt.xlabel('Income', fontsize=10)
plt.ylabel('Spent', fontsize=10)

plt.show()

K-means ha encontrado dos clúster con las siguientes características, mirando la gráfica anterior:
* Hay un cluster que concentra los consumidores que más gastan, los cuales en general tienen unos ingresos superiores
* El otro cluster concentra los consumidores con menores gastos.


### Extra: Encontrar el número óptimo de clústers mediante el Silhouette Score 

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []

# Evalúo el algoritmo K-means para un rango de [2,15] clústers 
n_cluster_list = range(2,15)

In [ ]:
for n_cluster in n_cluster_list:
    kmeans = KMeans(n_clusters=n_cluster, random_state=0)
    cluster_found = kmeans.fit_predict(X_scale)
    silhouette_scores.append(silhouette_score(X_scale, kmeans.labels_, random_state=0))

silhouette_metric = pd.DataFrame(index=n_cluster_list, columns=['silhouette_score'], data=silhouette_scores)
plt.plot(silhouette_metric, marker='o')

In [ ]:
# Entreno el K-means para k=4, ya que el resultado no es malo del todo y queremos probar un número de clústers distinto a 2
################ CÓDIGO AQUÍ ########################

In [ ]:
# Hacemos scatter plot entre 'Income' y 'Spent'
################ CÓDIGO AQUÍ ########################

## Validar los resultados con Dimensionality Reduction (PCA)
Principal Component Analysis (PCA) es un método estadístico que permite simplificar la complejidad de espacios muestrales con muchas dimensiones a la vez que conserva su información. En este caso, podemos trabajar con el dataset original (previo a reducirlo), ya que con este método se reducen las "features" conservando el máximo de información.

Repetimos los pasos seguidos anteriormente pero con todas las variables.

Una forma de validar los resultados del algoritmo clustering es mediante técnicas de dimensionality reduction. Hay que tener en cuenta es que el PCA no sabe nada de los grupos encontrados por K-means.

In [ ]:
df_cluster.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = df_cluster.values.copy()
X_scale = pd.DataFrame(scaler.fit_transform(X))
X_scale

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.colors

pca = PCA(n_components=2)
results_pca = pca.fit_transform(X_scale)

plt.scatter(results_pca[:, 0], results_pca[:, 1],
            c = df_cluster_def['cluster'],
            cmap = 'cool'
            )
plt.show()